# Chapter 1 — Exercise Solutions## What Is Reinforcement Learning?[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)> **Note:** Attempt the exercises in `01_exercises.ipynb` first. Some blanks are left for you to fill — marked `# YOUR TURN`.

### Solution 1.1: GridWorld with Teleport

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import defaultdict

class GridWorldTeleport:
    """GridWorld where hitting a wall teleports the agent back to (0,0)."""
    def __init__(self, size=5):
        self.size = size
        self.goal = (size-1, size-1)
        self.walls = {(1,1), (1,2), (2,2), (3,1)}
        self.actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']
        self.action_map = {
            'UP': (-1, 0), 'DOWN': (1, 0),
            'LEFT': (0, -1), 'RIGHT': (0, 1)
        }
        self.reset()

    def reset(self):
        self.pos = (0, 0)
        return self.pos

    def step(self, action):
        dr, dc = self.action_map[action]
        new_r = self.pos[0] + dr
        new_c = self.pos[1] + dc

        # KEY CHANGE: teleport to (0,0) instead of staying
        if (0 <= new_r < self.size and
            0 <= new_c < self.size and
            (new_r, new_c) not in self.walls):
            self.pos = (new_r, new_c)
            reward = -1   # normal step cost
        else:
            self.pos = (0, 0)   # TELEPORT back to start
            reward = -5         # penalty for hitting wall/boundary

        if self.pos == self.goal:
            reward = +10
            done = True
        else:
            done = False

        return self.pos, reward, done

# ── Training helper ──────────────────────────────────────────────
class SimpleQLearner:
    def __init__(self, actions, alpha=0.3, gamma=0.9, epsilon=0.3):
        self.actions = actions
        self.alpha, self.gamma, self.epsilon = alpha, gamma, epsilon
        self.Q = defaultdict(lambda: defaultdict(float))

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.choice(self.actions)
        return max(self.actions, key=lambda a: self.Q[state][a])

    def update(self, s, a, r, s_next, done):
        best_next = max(self.Q[s_next][a2] for a2 in self.actions)
        target = r + (0 if done else self.gamma * best_next)
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

def train(EnvClass, n_episodes=2000):
    agent = SimpleQLearner(actions=['UP','DOWN','LEFT','RIGHT'])
    rewards = []
    for ep in range(n_episodes):
        env = EnvClass()
        state = env.reset()
        ep_reward = 0
        for _ in range(100):
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            ep_reward += reward
            state = next_state
            if done: break
        rewards.append(ep_reward)
        agent.epsilon = max(0.05, agent.epsilon * 0.999)
    return rewards

# Train both environments and compare
from chapter_01_what_is_rl import GridWorld  # or redefine inline
# For standalone running, inline original GridWorld:
rewards_original  = train(lambda: __import__('types').SimpleNamespace(
    size=5, goal=(4,4), walls={(1,1),(1,2),(2,2),(3,1)},
    pos=(0,0),
    actions=['UP','DOWN','LEFT','RIGHT'],
    action_map={'UP':(-1,0),'DOWN':(1,0),'LEFT':(0,-1),'RIGHT':(0,1)},
    reset=lambda self: (setattr(self,'pos',(0,0)) or self.pos),
    step=None  # placeholder
))

# Simpler: run both directly
rewards_teleport = train(GridWorldTeleport)
print(f"Teleport env — final 100-ep mean: {np.mean(rewards_teleport[-100:]):.2f}")
print("Expected: slower initial learning (teleporting resets all progress)")
print("But can converge to comparable final performance once agent learns to avoid walls")

In [ ]:
# YOUR TURN: Run both and plot the comparison
# window = 50
# smoothed_teleport = np.convolve(rewards_teleport, np.ones(window)/window, mode='valid')
# Plot both smoothed curves on the same axes
# Label clearly, add legend and grid
# Answer: teleport version typically shows higher early variance
#         but may converge faster once agent learns to fear walls


**Key insight:** Teleporting resets all progress, so the agent learns wall-avoidance more urgently. The learning curve is more volatile early but the penalty signal is clearer.

### Solution 1.2: RL Loop → A/B Test Mapping

In [ ]:
# Complete mapping: RL concepts ↔ A/B testing concepts
answers = {
    'User who sees a page':       'Environment  (the thing the agent interacts with)',
    'Button variant (A or B)':    'Action       (the choice taken at each step)',
    'Conversion (click)':         'Reward       (the scalar feedback signal)',
    'User context (device, time)':'State        (the observation used to choose action)',
    'Test running for 2 weeks':   'Episode / Training run (the full interaction period)',
}

print("A/B Test Concept          →  RL Equivalent")
print("-" * 60)
for concept, rl_term in answers.items():
    print(f"  {concept:<30} → {rl_term}")

# YOUR TURN: Add two more mappings of your own:
# e.g. "Statistical significance threshold" → ???
# e.g. "Holdout control group"              → ???


### Solution 1.3: Exploration-Exploitation Tradeoff

In [ ]:
import numpy as np, random
from collections import defaultdict

def train_with_epsilon(epsilon_start=0.4, n_episodes=2000):
    size = 5
    goal = (4, 4)
    walls = {(1,1),(1,2),(2,2),(3,1)}
    actions = ['UP','DOWN','LEFT','RIGHT']
    action_map = {'UP':(-1,0),'DOWN':(1,0),'LEFT':(0,-1),'RIGHT':(0,1)}

    Q = defaultdict(lambda: defaultdict(float))
    epsilon = epsilon_start
    rewards = []

    for ep in range(n_episodes):
        pos = (0, 0)
        ep_reward = 0
        for _ in range(50):
            # Action selection
            if random.random() < epsilon:
                action = random.choice(actions)
            else:
                action = max(actions, key=lambda a: Q[pos][a])

            dr, dc = action_map[action]
            nr, nc = pos[0]+dr, pos[1]+dc
            if 0<=nr<size and 0<=nc<size and (nr,nc) not in walls:
                npos = (nr, nc)
                r = 10 if npos==goal else -1
            else:
                npos = pos
                r = -5

            best_next = max(Q[npos][a] for a in actions)
            target = r + (0 if npos==goal else 0.9 * best_next)
            Q[pos][action] += 0.3 * (target - Q[pos][action])

            ep_reward += r
            pos = npos
            if pos == goal: break

        rewards.append(ep_reward)
        # Decay only if epsilon > 0
        if epsilon_start > 0:
            epsilon = max(0.05, epsilon * 0.999)

    return rewards

# Run three variants
r_zero    = train_with_epsilon(epsilon_start=0.0)   # pure exploitation
r_normal  = train_with_epsilon(epsilon_start=0.4)   # normal
r_high    = train_with_epsilon(epsilon_start=1.0)   # pure exploration then decay

window = 50
fig, ax = plt.subplots(figsize=(12, 4))
for rewards, label, color in [
    (r_zero,   'ε=0.0 (pure exploit)', 'tomato'),
    (r_normal, 'ε=0.4 (normal)',       'steelblue'),
    (r_high,   'ε=1.0 → decay',        'seagreen'),
]:
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=label, color=color, linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed Reward')
ax.set_title('Effect of Exploration Rate on Learning')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nε=0.0 result:", np.mean(r_zero[-100:]))
print("Expected: stays near initial random policy — never discovers the goal reliably")
print("ε=0.4 result:", np.mean(r_normal[-100:]))
print("ε=1.0 result:", np.mean(r_high[-100:]))

# YOUR TURN: Find the minimum epsilon that still allows learning
# Try: 0.001, 0.01, 0.05, 0.1 — which is the crossover point?


### Solution 1.4: Custom Reward Functions

In [ ]:
# Three reward function designs with predictions

# Design 1: Speed-maximising (heavy step penalty)
def reward_speed(pos, npos, goal, hit_wall):
    if hit_wall: return -1        # mild wall penalty
    if npos == goal: return +100  # big goal bonus
    return -5                     # heavy step penalty → agent rushes

# Design 2: Wall-cautious (catastrophic wall penalty)
def reward_cautious(pos, npos, goal, hit_wall):
    if hit_wall: return -50       # catastrophic wall penalty
    if npos == goal: return +10
    return -0.1                   # tiny step penalty → patience encouraged

# Design 3: Exploration-encouraging (distance bonus)
def reward_explore(pos, npos, goal, hit_wall):
    if hit_wall: return -5
    if npos == goal: return +10
    # Bonus for getting closer to goal (Manhattan distance)
    dist_before = abs(pos[0]-goal[0]) + abs(pos[1]-goal[1])
    dist_after  = abs(npos[0]-goal[0]) + abs(npos[1]-goal[1])
    shaping = dist_before - dist_after  # +1 if closer, -1 if farther
    return -0.5 + shaping

print("Reward function predictions:")
print("  Speed:     agent takes shortest path, high variance early")
print("  Cautious:  agent learns to hug safe paths, slower but stable")
print("  Explore:   agent guided toward goal by shaping signal, fastest convergence")

# YOUR TURN: Implement the training loop for reward_cautious
# Use the GridWorld from the concept notebook but replace step rewards
# Compare learning curves for all three
